In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1b605755-f1f5-4dad-9df9-d56c676132f4;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 368ms :: artifacts dl 20ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers_path = "s3a://last-mile-optimization-trusted/dataset-orders/customers_cleaned_dataset.csv/"
geolocation_path = "s3a://last-mile-optimization-trusted/dataset-orders/geolocation_cleaned_dataset.csv/"

customers_df = spark.read.csv(customers_path, header=True, inferSchema=True)
geolocation_df = spark.read.csv(geolocation_path, header=True, inferSchema=True)

# Create an index per zip for both tables and map customers to geolocations by modulo
geo_w = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.monotonically_increasing_id())
geo_indexed = (
    geolocation_df
    .withColumn("geo_index", F.row_number().over(geo_w))
    .withColumn("geo_count", F.count("*").over(Window.partitionBy("geolocation_zip_code_prefix")))
    .withColumnRenamed("geolocation_zip_code_prefix", "customer_zip_code_prefix")
)

geo_counts = geo_indexed.select("customer_zip_code_prefix", "geo_count").dropDuplicates(["customer_zip_code_prefix"])

cust_w = Window.partitionBy("customer_zip_code_prefix").orderBy(F.monotonically_increasing_id())
customers_indexed = customers_df.withColumn("cust_index", F.row_number().over(cust_w))

customers_with_geo_count = customers_indexed.join(
    geo_counts,
    on="customer_zip_code_prefix",
    how="inner"
)

customers_with_geo_index = customers_with_geo_count.withColumn(
    "geo_index",
    F.pmod(F.col("cust_index") - F.lit(1), F.col("geo_count")) + F.lit(1)
)

joined_df = customers_with_geo_index.join(
    geo_indexed,
    on=["customer_zip_code_prefix", "geo_index"],
    how="inner"
)

joined_df = joined_df.drop("geo_count", "geo_index", "cust_index")

joined_df.show(5)
print(f"Total rows: {joined_df.count()}")

26/04/04 23:03:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+------------------------+--------------------+--------------------+-------------+-------------------+-------------------+
|customer_zip_code_prefix|         customer_id|  customer_unique_id|customer_city|    geolocation_lat|    geolocation_lng|
+------------------------+--------------------+--------------------+-------------+-------------------+-------------------+
|                    1226|a32e9f128bf594a9f...|04964ca17488b7612...|    SAO PAULO|-23.538190850683794|-46.651323227306854|
|                    1238|89aac25164a5f84e4...|3ac427a63b9d1586f...|    SAO PAULO| -23.54195269441404| -46.65931762155928|
|                    1311|368cbdd696020410f...|5bfb289793467d2fc...|    SAO PAULO|-23.571018376363245| -46.64477790550516|
|                    2072|882ee7ad589b74292...|ddd6924bb0a8101c1...|    SAO PAULO|-23.499949801192507| -46.59311388119833|
|                    2531|6c2d4f0f802bfa13d...|18256891fed9a64ad...|    SAO PAULO|-23.498286024150236| -46.65922176395056|
+---------------

Total rows: 41731


In [4]:
joined_df.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-orders/join_customers_and_geolocation.csv')

spark.stop()

26/04/04 23:04:10 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/04 23:04:11 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
